# 🌐 Язык / Language

**RU.** Выберите язык в ячейке ниже: `LANG = "ru"` или `LANG = "en"`, затем запустите блокнот целиком (Run All).
На выбранном языке будет весь текст: пояснения, выводы ячеек, заголовки, подписи и легенды графиков.

**EN.** Choose the language in the cell below: `LANG = "ru"` or `LANG = "en"`, then run the whole notebook (Run All).
Everything will be in that language: the explanations, the cell outputs, chart titles, labels and legends.

In [ ]:
LANG = "ru"   # "ru" -- русский | "en" -- English

# Autoreload: edits to transform.py are picked up without restarting the kernel.
# Автоперезагрузка: правки transform.py подхватываются без перезапуска ядра.
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

import transform as tf
tf.set_lang(LANG)
tr, show_md = tf.tr, tf.show_md
pal = tf.econ_style('light')
print(sys.executable)
print(tr('Язык блокнота: русский. Модуль transform, версия', 'Notebook language: English. Module transform, version'),
      tf.__version__, '|', tf.__file__)

In [ ]:
show_md({
"ru": r"""
# Сырые данные: размерности индикаторов и как из одних получать другие

Третий блокнот серии после сезонной корректировки и выделения тренда. Те два блокнота отвечают на вопрос «что делать
с рядом», этот — на более ранний: **что вообще лежит в ряду и во что его можно превратить**.

Каждый индикатор — это таблица `Date, Value` плюс **характеристики**: что за величина, в каком виде опубликована, в чём
измерена, запас это или поток, в текущих ли ценах. Одно и то же число `1,2` может быть приростом цен за месяц,
триллионами рублей с начала года или долей безработных. Характеристики говорят, что это за число, а значит — какие
преобразования осмысленны.

Все функции — в модуле `transform.py` рядом с блокнотом. Блокнот только объясняет и вызывает.

**План**
1. Настройки, характеристики, загрузка
2. Какие характеристики встречаются
3. Теория и рецепты: единицы и `mult`; запас, поток, среднее; `metric` и п.п.; `transf`; реальное и номинальное; валюты; %ВВП
4. Карта преобразований: что из чего получается, а что — нет (с доказательствами)
5. Один график — все метрики одного индикатора
6. Живые примеры по типам показателей
7. Свой расчёт: любой тикер, любая функция
8. Шпаргалка и типичные ошибки
""",
"en": r"""
# Raw data: indicator dimensions and how to get one from another

The third notebook of the series, after seasonal adjustment and trend extraction. Those two answer the question "what to
do with a series"; this one answers an earlier question: **what the series actually holds and what it can be turned into**.

Every indicator is a `Date, Value` table plus **characteristics**: what kind of quantity it is, in what form it is
published, in which units, whether it is a stock or a flow, whether it is in current prices. The same number `1.2` can be
a monthly price increase, trillions of roubles since January or the unemployment share. The characteristics say what the
number is — and therefore which transformations make sense.

All functions live in the `transform.py` module next to the notebook. The notebook only explains and calls them.

**Plan**
1. Settings, characteristics, loading
2. Which characteristics occur
3. Theory and recipes: units and `mult`; stock, flow, average; `metric` and p.p.; `transf`; real vs nominal; currencies; % of GDP
4. The transformation map: what can be derived from what — and what cannot (with proofs)
5. One figure — every metric of one indicator
6. Live examples by indicator type
7. Your own calculation: any ticker, any function
8. Cheat sheet and typical mistakes
"""})

In [ ]:
show_md({
"ru": r"""
## 1. Настройки и загрузка

Данные читаются из Excel-файлов: один тикер — один файл `<ТИКЕР>.xlsx` (или `.csv`) с колонками `Date, Value`
в папке `DATA_DIR`. Характеристики каждого ряда описываются вручную (1.2). Чтобы добавить свой ряд, положите файл
в ту же папку и допишите его характеристики.
""",
"en": r"""
## 1. Settings and loading

Data are read from Excel files: one ticker — one file `<TICKER>.xlsx` (or `.csv`) with columns `Date, Value` in the
`DATA_DIR` folder. The characteristics of each series are described by hand (1.2). To add your own series, put the file
in the same folder and add its characteristics.
"""})

In [ ]:
DATA_DIR    = 'RAW_DATA'         # folder with <TICKER>.xlsx files | папка с файлами <ТИКЕР>.xlsx

GDP_Q       = 'RU_GDP_Q_RS'      # nominal GDP, quarterly | номинальный ВВП, квартальный
GDP_REAL_Q  = 'RU_GDPR_Q_RS'     # real GDP, quarterly (constant prices) | реальный ВВП, квартальный (в постоянных ценах)
BASE_CCY    = 'RUB'              # currency of GDP | валюта ВВП
FX_SUFFIXES = ['D', 'CBR']       # FX: generic (_D) first, then the official CBR rate (_CBR) | сначала generic, потом ЦБ
START       = '2015-01-01'       # charts start from | с какой даты показывать графики

# Manual choice of tickers for the examples of section 6 (keys -- indicator types, see 1.3), e.g.:
# Ручной выбор тикеров для примеров раздела 6 (ключи -- типы показателей, см. 1.3), например:
# TYPE_OVERRIDES = {'cur_flow_cmlt': 'RU_KBK_0_FED_M_FT'}
TYPE_OVERRIDES = {}

In [ ]:
show_md({
"ru": r"""
### 1.2 Характеристики индикаторов

| поле | вопрос | значения |
|---|---|---|
| `metric` | что за величина? | `index` — число «как есть» (уровень или прирост); `%` — доля; `rate` — ставка, % годовых; `%gdp` — уже отнесено к ВВП |
| `transf` | в каком виде опубликовано? | `none` — уровень; `cmlt` — с начала года; `mom`/`qoq`/`wow` — к предыдущему периоду; `yoy` — к тому же периоду год назад; `ytd` — к концу прошлого года; `cmlt_cmlt` — с начала года к тому же периоду прошлого года |
| `dim` | в чём измеряется уровень? | `currency`, `items` (шт.), `km`, `m`, `m2`, `m3`, `t`, `persons`, `none` |
| `currency` | если `dim = currency` — в какой валюте? | `RUB`, `USD`, `KZT`, … |
| `mult` | масштаб хранения | 1 — единицы, 10³ — тысячи, 10⁶ — миллионы, 10⁹ — миллиарды |
| `stock_flow` | запас или поток? | `stock` — снимок на дату; `flow` — сумма за период |
| `eop_avg` | для запасов и цен: на конец или в среднем? | `eop`, `avg` |
| `freq` | частота | `D`, `W`, `M`, `Q`, `Y` |
| `real_nominal` | в текущих или постоянных ценах? | `nominal`, `real`, `none` |
| `sa`, `ca` | очищен ли от сезонности / календаря | True / False |

Характеристики заполняем **вручную** в ячейке ниже — по одному словарю на тикер; тикер = имя файла без `.xlsx`. Названия — на двух языках (`short_name_rus`, `short_name_en`).
Проверьте `mult` и `real_nominal`.
""",
"en": r"""
### 1.2 Indicator characteristics

| field | question | values |
|---|---|---|
| `metric` | what kind of quantity? | `index` — the number "as is" (level or growth); `%` — a share; `rate` — a rate, % per annum; `%gdp` — already a ratio to GDP |
| `transf` | in what form is it published? | `none` — level; `cmlt` — since January; `mom`/`qoq`/`wow` — vs previous period; `yoy` — vs the same period a year earlier; `ytd` — vs the end of last year; `cmlt_cmlt` — year-to-date vs the same period last year |
| `dim` | what are the units of the level? | `currency`, `items`, `km`, `m`, `m2`, `m3`, `t`, `persons`, `none` |
| `currency` | if `dim = currency` — which currency? | `RUB`, `USD`, `KZT`, … |
| `mult` | storage scale | 1 — units, 10³ — thousands, 10⁶ — millions, 10⁹ — billions |
| `stock_flow` | stock or flow? | `stock` — a snapshot at a date; `flow` — a sum over the period |
| `eop_avg` | for stocks and prices: end of period or average? | `eop`, `avg` |
| `freq` | frequency | `D`, `W`, `M`, `Q`, `Y` |
| `real_nominal` | current or constant prices? | `nominal`, `real`, `none` |
| `sa`, `ca` | seasonally / calendar adjusted? | True / False |

The characteristics are filled in **by hand** in the cell below — one dict per ticker; the ticker is the file name
without `.xlsx`. Names are given in both languages
(`short_name_rus`, `short_name_en`). Check `mult` and `real_nominal`.
"""})

In [ ]:
MANUAL_CHARACTERISTICS = [
    dict(ticker='FX_RUBUSD_D', short_name_rus='Курс рубля к доллару', short_name_en='RUB per USD',
         metric='index', transf='none', dim='currency', currency='RUB', mult=1,
         stock_flow='none', eop_avg='avg', freq='D', real_nominal='nominal', source_org='generic'),
    dict(ticker='RATE_RU_KEYRATE_CBR', short_name_rus='Ключевая ставка', short_name_en='Key rate',
         metric='rate', transf='none', dim='none', currency='none', mult=1,
         stock_flow='none', eop_avg='eop', freq='D', real_nominal='nominal', source_org='Bank of Russia'),
    dict(ticker='RU_CP_1820_M_RS', short_name_rus='Добыча газа', short_name_en='Natural gas production',
         metric='index', transf='none', dim='m3', currency='none', mult=1e6,
         stock_flow='flow', freq='M', real_nominal='real', source_org='Rosstat'),
    dict(ticker='RU_CPIMOM_CC_0_M_RS', short_name_rus='ИПЦ, м/м', short_name_en='CPI, m/m',
         metric='index', transf='mom', dim='none', currency='none', mult=1,
         stock_flow='none', eop_avg='avg', freq='M', real_nominal='none', source_org='Rosstat'),
    dict(ticker='RU_CPIWOW_CC_0_W_RS', short_name_rus='ИПЦ, н/н', short_name_en='CPI, w/w',
         metric='index', transf='wow', dim='none', currency='none', mult=1,
         stock_flow='none', eop_avg='avg', freq='W', real_nominal='none', source_org='Rosstat'),
    dict(ticker='RU_EXPG_Q_CB', short_name_rus='Экспорт товаров', short_name_en='Exports of goods',
         metric='index', transf='none', dim='currency', currency='USD', mult=1e6,
         stock_flow='flow', freq='Q', real_nominal='nominal', source_org='Bank of Russia'),
    dict(ticker='RU_GDP_Q_RS', short_name_rus='ВВП номинальный', short_name_en='Nominal GDP',
         metric='index', transf='none', dim='currency', currency='RUB', mult=1e9,
         stock_flow='flow', freq='Q', real_nominal='nominal', source_org='Rosstat'),
    dict(ticker='RU_GDPR_Q_RS', short_name_rus='ВВП реальный', short_name_en='Real GDP',
         metric='index', transf='none', dim='currency', currency='RUB', mult=1e9,
         stock_flow='flow', freq='Q', real_nominal='real', source_org='Rosstat'),
    dict(ticker='RU_INDYOY_NC_2882_M_RS', short_name_rus='Промышленное производство, г/г',
         short_name_en='Industrial production, y/y',
         metric='index', transf='yoy', dim='none', currency='none', mult=1,
         stock_flow='flow', freq='M', real_nominal='real', source_org='Rosstat'),
    dict(ticker='RU_INVCMLT_Q_RS', short_name_rus='Инвестиции в основной капитал, с начала года к прошлому году',
         short_name_en='Fixed investment, year-to-date vs last year',
         metric='index', transf='cmlt_cmlt', dim='none', currency='none', mult=1,
         stock_flow='flow', freq='Q', real_nominal='real', source_org='Rosstat'),
    dict(ticker='RU_KBK_0_FED_M_FT', short_name_rus='Доходы федерального бюджета, с начала года',
         short_name_en='Federal budget revenue, year-to-date',
         metric='index', transf='cmlt', dim='currency', currency='RUB', mult=1,
         stock_flow='flow', freq='M', real_nominal='nominal', source_org='Treasury'),
    dict(ticker='RU_M2XXFX_M_CB', short_name_rus='Денежная масса M2X без валютной переоценки, с начала года',
         short_name_en='Money supply M2X ex FX revaluation, year-to-date',
         metric='index', transf='ytd', dim='none', currency='none', mult=1,
         stock_flow='stock', eop_avg='eop', freq='M', real_nominal='nominal', source_org='Bank of Russia'),
    dict(ticker='RU_RES_M_CB', short_name_rus='Международные резервы', short_name_en='International reserves',
         metric='index', transf='none', dim='currency', currency='USD', mult=1e9,
         stock_flow='stock', eop_avg='eop', freq='M', real_nominal='nominal', source_org='Bank of Russia'),
    dict(ticker='RU_UNEMPLR_M_RS', short_name_rus='Уровень безработицы', short_name_en='Unemployment rate',
         metric='%', transf='none', dim='none', currency='none', mult=1,
         stock_flow='none', eop_avg='avg', freq='M', real_nominal='none', source_org='Rosstat'),
]

In [ ]:
CHARS = tf.characteristics_from_records(MANUAL_CHARACTERISTICS)
print(tr(f'Файлы в {DATA_DIR}: характеристики {len(CHARS)} индикаторов',
         f'Files in {DATA_DIR}: characteristics of {len(CHARS)} indicators'))
no_chars = sorted({os.path.splitext(f)[0] for f in os.listdir(DATA_DIR) if f.endswith(('.xlsx', '.csv'))}
                  - set(CHARS.index))
no_file = sorted(t for t in CHARS.index if tf.file_read_raw(t, DATA_DIR) is None)
if no_file:
    print(tr('!!! Характеристики без файла (тикер должен совпадать с именем файла):',
             '!!! Characteristics without a file (the ticker must match the file name):'), no_file)
if no_chars:
    print(tr('!!! Файлы без характеристик (добавьте в MANUAL_CHARACTERISTICS):',
             '!!! Files without characteristics (add them to MANUAL_CHARACTERISTICS):'), no_chars)

CHARS['name'] = [tf.name_of(tf.get_chars(CHARS, t)) for t in CHARS.index]
cols = ['name', 'metric', 'transf', 'dim', 'currency', 'mult', 'stock_flow', 'eop_avg', 'freq', 'real_nominal']
display(CHARS[cols].sort_index() if len(CHARS) < 60 else CHARS[cols].head(60))

In [ ]:
show_md({
"ru": r"""
### 1.3 Типы показателей

Тип — это типичное сочетание характеристик: «показатель в валюте, поток, номинальный», «прирост к предыдущему месяцу»
и т.д. Для каждого типа блокнот подбирает пример из описанных рядов; любой можно заменить через `TYPE_OVERRIDES`.
**Тип нужен только для примеров раздела 6.** Считать можно что угодно и без него — раздел 7.
""",
"en": r"""
### 1.3 Indicator types

A type is a typical combination of characteristics: "money indicator, flow, nominal", "growth vs previous month" and so on.
For each type the notebook picks an example from the described series; any of them can be replaced via `TYPE_OVERRIDES`.
**The type is needed only for the examples of section 6.** Anything can be computed without it — section 7.
"""})

In [ ]:
EX = {}
for key in tf.INDICATOR_TYPES:
    EX[key] = TYPE_OVERRIDES.get(key) or tf.find_example(CHARS, key, BASE_CCY)
display(pd.DataFrame([(k, tf.type_title(k), EX[k] or tr('— не найден —', '— not found —'),
                       CHARS.loc[EX[k], 'name'] if EX[k] in CHARS.index else '')
                      for k in tf.INDICATOR_TYPES],
                     columns=[tr('ключ', 'key'), tr('тип показателя', 'indicator type'), tr('тикер', 'ticker'),
                              tr('название', 'name')]))

In [ ]:
show_md({
"ru": r"""
### 1.4 Загрузка рядов, курсов и ВВП

Курсы хранятся как «сколько единиц валюты за 1 доллар»: `FX_RUBUSD_D` ≈ 80 — это 80 рублей за доллар.
Доллар — общий знаменатель: любая валюта A переводится в B через него, $x_B = x_A / (A\text{ за USD}) \times (B\text{ за USD})$.
Поэтому достаточно одного ряда `FX_<ВАЛ>USD_*` на валюту.
""",
"en": r"""
### 1.4 Loading series, FX rates and GDP

FX rates are stored as "units of currency per 1 dollar": `FX_RUBUSD_D` ≈ 80 means 80 roubles per dollar.
The dollar is the common denominator: any currency A is converted to B through it, $x_B = x_A / (A\text{ per USD}) \times (B\text{ per USD})$.
So one `FX_<CCY>USD_*` series per currency is enough.
"""})

In [ ]:
SER = {}


def C(t):
    """Characteristics of a ticker as a dict. | Характеристики тикера словарём."""
    return tf.get_chars(CHARS, t)


def N(t):
    """Indicator name in the current language. | Название индикатора на текущем языке."""
    return tf.name_of(C(t))


def load(t):
    """Load a ticker once and keep it in SER. | Загружает тикер один раз и кладёт в SER."""
    if t in SER:
        return SER[t]
    c = C(t)
    if c is None:
        print(tr(f'  нет характеристик: {t}', f'  no characteristics: {t}'))
        return None
    s = tf.load_series(t, c['freq'], folder=DATA_DIR)
    if s is None:
        print(tr(f'  нет данных: {t}', f'  no data: {t}'))
        return None
    SER[t] = s
    return s


for k, t in EX.items():
    if t and load(t) is None:
        EX[k] = None
load(GDP_Q)
load(GDP_REAL_Q)

need_ccy = {BASE_CCY} | {C(t)['currency'] for t in SER if C(t)['dim'] == 'currency'}
need_ccy -= {'USD', 'none'}
FX_TICKER = {}
for ccy in sorted(need_ccy):
    for suf in FX_SUFFIXES:
        t = f'FX_{ccy}USD_{suf}'
        df = tf.file_read_raw(t, DATA_DIR)
        if df is not None and not df.empty:
            tf.set_fx(ccy, tf.to_series(df, 'D'))
            FX_TICKER[ccy] = t
            break
    else:
        print(tr(f'!!! курс для {ccy} не найден', f'!!! no FX rate found for {ccy}'))

rows = [(t, C(t)['freq'], s.dropna().index[0], s.dropna().index[-1], s.notna().sum()) for t, s in SER.items()]
rows += [(FX_TICKER[c], 'D', s.index[0].date(), s.index[-1].date(), len(s)) for c, s in tf.FX.items()]
display(pd.DataFrame(rows, columns=[tr('тикер', 'ticker'), tr('частота', 'freq'), tr('с', 'from'), tr('по', 'to'),
                                    tr('наблюдений', 'observations')]))

GDP_UNITS = tf.to_units(SER[GDP_Q], C(GDP_Q))           # quarterly GDP in roubles | квартальный ВВП в рублях

In [ ]:
show_md({
"ru": r"""
## 2. Какие характеристики встречаются

Поля почти независимы: «индекс, г/г, валюта, рубли, млрд, поток, месячный, номинальный» — это одна строка характеристик.
""",
"en": r"""
## 2. Which characteristics occur

The fields are almost independent: "index, y/y, currency, roubles, billions, flow, monthly, nominal" is one row of characteristics.
"""})

In [ ]:
total = tr('всего', 'total')
print('metric × transf:')
display(pd.crosstab(CHARS['metric'], CHARS['transf'], margins=True, margins_name=total))
print('dim × stock_flow:')
display(pd.crosstab(CHARS['dim'], CHARS['stock_flow'], margins=True, margins_name=total))
for col in ['currency', 'freq', 'mult', 'real_nominal']:
    print(f'{col}:', CHARS[col].value_counts().to_dict())
unknown = sorted(set(CHARS['transf']) - tf.KNOWN_TRANSF) + sorted(set(CHARS['metric']) - set(tf.METRIC_ALIASES.values()))
if unknown:
    print(tr('!!! Значения, которых нет в словарях tf.TRANSF_ALIASES / tf.METRIC_ALIASES:',
             '!!! Values missing from tf.TRANSF_ALIASES / tf.METRIC_ALIASES:'), unknown)

In [ ]:
show_md({
"ru": r"""
## 3. Теория и рецепты

### 3.1 Единицы: `mult`, `dim`, `currency`

В таблице лежит `Value`, а реальная величина — `Value × mult` в единицах `dim` (и валюте `currency`).
Даже в одной папке масштабы разные: доходы бюджета от Казначейства — в рублях (`mult = 1`), ВВП — в миллиардах рублей,
экспорт по платёжному балансу — в миллионах долларов, добыча газа — в миллионах м³.
**Перед любым делением одного ряда на другой оба переводим в абсолютные единицы** — иначе ошибка в тысячу раз.
""",
"en": r"""
## 3. Theory and recipes

### 3.1 Units: `mult`, `dim`, `currency`

The table holds `Value`; the actual quantity is `Value × mult` in units of `dim` (and currency `currency`).
Even within one folder the scales differ: Treasury budget revenue is in roubles (`mult = 1`), GDP in billions of roubles,
balance-of-payments exports in millions of dollars, gas production in millions of m³.
**Before dividing one series by another, convert both to absolute units** — otherwise the error is a factor of a thousand.
"""})

In [ ]:
tf.set_section('3.1', {'ru': 'Единицы', 'en': 'Units'})
rows = []
for t in SER:
    c = C(t)
    if c['metric'] == 'index' and c['transf'] in ('none', 'cmlt'):
        v = SER[t].dropna()
        rows.append((t, N(t), tf.ru_num(v.iloc[-1], 2), f'{c["mult"]:g}',
                     tf.fmt_amount(tf.to_units(v.iloc[-1], c), tf.unit_label(c))))
display(pd.DataFrame(rows, columns=[tr('тикер', 'ticker'), tr('название', 'name'), tr('значение в данных', 'stored value'),
                                    'mult', tr('в абсолютных единицах', 'in absolute units')]))

In [ ]:
show_md({
"ru": r"""
### 3.2 Запас, поток и «среднее за период»

| | **поток** (`flow`) | **запас** (`stock`) | **среднее за период** (`avg`) |
|---|---|---|---|
| что это | сколько произошло **за** период | сколько есть **на** дату | средний уровень за период |
| примеры | ВВП, доходы бюджета, экспорт, добыча | резервы, денежная масса, долг | цены, курс, ставка |
| месяц → квартал | **сумма** трёх месяцев | значение на **конец** квартала | **среднее** трёх месяцев |
| скользящее окно | сумма за 12 мес. (годовой объём) или среднее | среднее (средний запас) | среднее |
| перевод в другую валюту | по **среднему** курсу периода | по курсу на **конец** периода | по среднему курсу |
| к ВВП | квартал к кварталу ВВП или 4 кв. к 4 кв. | на конец квартала к **годовому** ВВП | обычно не относят |

Почему курс разный: поток набирался весь период по разным курсам, и средний курс — лучшее приближение к курсу, по которому
он «случился». Запас существует в конкретный момент, и его стоимость в другой валюте определяется курсом этого момента.

`tf.aggregate` сворачивает ряд по правилу из таблицы и **отбрасывает неполные периоды**: сумма двух месяцев квартала —
не квартальный поток, а последний месяц неполного квартала — не конец квартала.
""",
"en": r"""
### 3.2 Stock, flow and "period average"

| | **flow** (`flow`) | **stock** (`stock`) | **period average** (`avg`) |
|---|---|---|---|
| what it is | how much happened **over** the period | how much there is **at** a date | average level over the period |
| examples | GDP, budget revenue, exports, production | reserves, money supply, debt | prices, FX rate, interest rate |
| month → quarter | **sum** of three months | value at the **end** of the quarter | **average** of three months |
| rolling window | 12-month sum (annual volume) or mean | mean (average stock) | mean |
| to another currency | at the **average** rate of the period | at the **end-of-period** rate | at the average rate |
| to GDP | quarter to quarterly GDP or 4Q to 4Q | end of quarter to **annual** GDP | usually not |

Why the rate differs: a flow accumulated over the whole period at different rates, and the average rate is the best
approximation of the rate at which it "happened". A stock exists at a specific moment, and its value in another currency
is set by the rate of that moment.

`tf.aggregate` reduces a series by the rule in the table and **drops incomplete periods**: two months of a quarter are
not a quarterly flow, and the last month of an incomplete quarter is not the end of the quarter.
"""})

In [ ]:
fx = tf.FX['RUB']
c_avg, c_eop, c_d = tr('средний за месяц', 'monthly average'), tr('на конец месяца', 'end of month'), tr('разница, %', 'difference, %')
fx_m = pd.DataFrame({c_avg: tf.aggregate(fx, 'M', 'mean', 'D'), c_eop: tf.aggregate(fx, 'M', 'last', 'D')})
fx_m[c_d] = 100 * (fx_m[c_eop] / fx_m[c_avg] - 1)
recent = fx_m['2014':]
print(tr(f'{FX_TICKER["RUB"]}: курс на конец месяца отличается от среднего за месяц до {recent[c_d].abs().max():.0f}% '
         f'(с 2014 г.); месяцы с наибольшим расхождением:',
         f'{FX_TICKER["RUB"]}: the end-of-month rate differs from the monthly average by up to {recent[c_d].abs().max():.0f}% '
         f'(since 2014); months with the largest gap:'))
display(recent.loc[recent[c_d].abs().nlargest(5).index].round(2))

In [ ]:
show_md({
"ru": r"""
### 3.3 Поле `metric`: уровень, доля, ставка, %ВВП — и что такое «изменение в п.п.»

* `index` — число «как есть»: уровень в единицах `dim` (рубли, м³, пункты индекса) или его прирост (если `transf ≠ none`).
  С этим видом работает вся арифметика ниже.
* `%` — доля (безработица, доля просрочки). Уже отношение, у него нет единиц.
* `rate` — ставка, % годовых. Как доля, но всегда «в год».
* `%gdp` — уже отнесено к ВВП.

**Изменение в процентных пунктах (п.п.)** — это разность двух процентных величин: $x_t - x_{t-1}$.
Безработица была 4,5%, стала 3,0% — это снижение на **1,5 п.п.** Относительное изменение той же доли — «на 33%» —
арифметически верно, но экономически почти бессмысленно и путает читателя.
Это не отдельное преобразование, а **способ считать «к предыдущему периоду» и «г/г» для величин, которые уже в процентах**:
для уровня мы делим ($L_t/L_{t-1} - 1$), для доли, ставки или темпа роста — вычитаем.
Поэтому в карте преобразований (раздел 4) для `%`, `rate`, `%gdp` колонки «к пред. периоду» и «г/г» помечены как
**«посчитать можно, но смысла нет» (◇)**: относительный прирост ставки или доли («ставка выросла на 135% г/г»)
ничего не значит, изменение считают разностью в п.п. (`tf.diff_pp`). То же — для разницы темпов роста
(«инфляция ускорилась с 4,1% до 5,3% г/г, на 1,2 п.п.»).
""",
"en": r"""
### 3.3 The `metric` field: level, share, rate, % of GDP — and what a "change in p.p." is

* `index` — the number "as is": a level in units of `dim` (roubles, m³, index points) or its growth (if `transf ≠ none`).
  All the arithmetic below works with this kind.
* `%` — a share (unemployment, the share of overdue loans). Already a ratio, it has no units.
* `rate` — a rate, % per annum. Like a share, but always "per year".
* `%gdp` — already a ratio to GDP.

**A change in percentage points (p.p.)** is the difference of two percentages: $x_t - x_{t-1}$.
Unemployment was 4.5% and became 3.0% — a fall of **1.5 p.p.** The relative change of the same share — "by 33%" — is
arithmetically correct but economically almost meaningless and confuses the reader.
It is not a separate transformation but **the way to compute "vs previous period" and "y/y" for quantities already in percent**:
for a level we divide ($L_t/L_{t-1} - 1$), for a share, rate or growth rate we subtract.
That is why in the transformation map (section 4) the "vs previous period" and "y/y" columns for `%`, `rate`, `%gdp` are marked
**"computable but meaningless" (◇)**: the relative growth of a rate or share ("the rate rose 135% y/y") means nothing;
the change is measured as a difference in p.p. (`tf.diff_pp`). The same applies to the difference of growth rates
("inflation accelerated from 4.1% to 5.3% y/y, by 1.2 p.p.").
"""})

In [ ]:
t = EX.get('share')
if t:
    u = SER[t].dropna()
    a, b = u.iloc[-13], u.iloc[-1]
    print(f'{N(t)}: {u.index[-13]} — {tf.ru_num(a, 2)}%, {u.index[-1]} — {tf.ru_num(b, 2)}%')
    print(tr(f'  изменение за год: {tf.ru_num(tf.diff_pp(u, 12).iloc[-1], 2)} п.п. '
             f'(а «в процентах от процента» было бы {tf.pct(100 * (b / a - 1), 1)} — так не говорят)',
             f'  change over the year: {tf.ru_num(tf.diff_pp(u, 12).iloc[-1], 2)} p.p. '
             f'(the "percent of a percent" would be {tf.pct(100 * (b / a - 1), 1)} — nobody says that)'))

In [ ]:
show_md({
"ru": r"""
### 3.4 Поле `transf`: в каком виде опубликован ряд

Пусть $L_t$ — уровень (поток за период, запас на дату или средний индекс цен), $k$ — число периодов в году.

| `transf` | формула | читается как |
|---|---|---|
| `none` | $L_t$ | «за март собрано 2,1 трлн руб.» |
| `cmlt` | $C_t = \sum_{\text{янв}}^{t} L$ внутри года | «за январь–март собрано 6,0 трлн руб.» |
| `mom`, `qoq`, `wow` | $100\,(L_t/L_{t-1}-1)$ | «цены за март выросли на 0,4%» |
| `yoy` | $100\,(L_t/L_{t-k}-1)$ | «производство в марте на 2,1% выше, чем в марте прошлого года» |
| `ytd` | $100\,(L_t/L_{\text{дек. прошлого года}}-1)$ | «с начала года денежная масса выросла на 3,6%» |
| `cmlt_cmlt` | $100\,(C_t/C_{t-k}-1)$ | «за январь–июнь инвестиции на 9,9% меньше, чем за январь–июнь прошлого года» |

**Формат приростов.** Росстат часто публикует приросты как индекс к базе 100: `100,4` вместо `0,4`.
`tf.growth_x` распознаёт формат по медиане ряда и всегда возвращает «чистый» прирост в процентах.

Проверки «туда и обратно» на настоящих данных — они же показывают, какие пары размерностей несут **одинаковую** информацию:
""",
"en": r"""
### 3.4 The `transf` field: in what form the series is published

Let $L_t$ be the level (a flow over the period, a stock at a date or an average price index), $k$ the number of periods per year.

| `transf` | formula | reads as |
|---|---|---|
| `none` | $L_t$ | "2.1 tn roubles collected in March" |
| `cmlt` | $C_t = \sum_{\text{Jan}}^{t} L$ within the year | "6.0 tn roubles collected in January–March" |
| `mom`, `qoq`, `wow` | $100\,(L_t/L_{t-1}-1)$ | "prices rose 0.4% in March" |
| `yoy` | $100\,(L_t/L_{t-k}-1)$ | "output in March was 2.1% higher than a year earlier" |
| `ytd` | $100\,(L_t/L_{\text{Dec last year}}-1)$ | "money supply is up 3.6% since the start of the year" |
| `cmlt_cmlt` | $100\,(C_t/C_{t-k}-1)$ | "investment in January–June was 9.9% lower than in January–June last year" |

**Growth format.** Rosstat often publishes growth as an index with base 100: `100.4` instead of `0.4`.
`tf.growth_x` detects the format from the median of the series and always returns plain growth in percent.

Round-trip checks on real data — they also show which pairs of dimensions carry **the same** information:
"""})

In [ ]:
checks = []
t = EX.get('cur_flow_cmlt')
if t:
    c_ = SER[t]
    checks.append((tr('cmlt → поток → cmlt', 'cmlt → flow → cmlt'), tf.same(tf.cumulate(tf.decumulate(c_)), c_)))
    lvl = tf.decumulate(c_)
    checks.append((tr('сумма 12 мес.: из потока = прямо из cmlt', '12-month sum: from the flow = directly from cmlt'),
                   tf.same(tf.rolling(lvl, 'M', 'Y', 'sum'), tf.roll12_from_cmlt(c_, 'M'))))
t = EX.get('pop_growth')
if t:
    x = tf.growth_x(SER[t])
    I = tf.chain(x)
    checks.append((tr('м/м → цепной индекс → м/м', 'm/m → chain index → m/m'), tf.same(tf.pop(I), x)))
    checks.append((tr('м/м → к декабрю (ytd) → м/м', 'm/m → vs December (ytd) → m/m'), tf.same(tf.mom_from_ytd(tf.ytd(I, 'M')), x)))
display(pd.DataFrame(checks, columns=[tr('проверка', 'check'), tr('совпало', 'matches')]))

In [ ]:
show_md({
"ru": r"""
### 3.5 Реальное и номинальное: поле `real_nominal`

**Номинальный** показатель (`nominal`) — в текущих ценах: ВВП в рублях того квартала, доходы бюджета, экспорт в долларах.
Он растёт и когда растёт объём, и когда растут цены.
**Реальный** (`real`) — в постоянных ценах базового года (реальный ВВП «в ценах 2021 г.») или в натуральных единицах
(м³ газа, тонны). Рост цен из него исключён.

Разница между ними — **цены**:

$$\text{дефлятор}_t = 100 \cdot \frac{\text{номинал}_t}{\text{реальный}_t}, \qquad
(1+n) = (1+r)(1+p) \;\Rightarrow\; n = r + p + \frac{r\,p}{100}$$

где $n$ — номинальный рост, $r$ — реальный, $p$ — рост цен (дефлятора). При небольших темпах $n \approx r + p$;
перекрёстный член заметен только при высокой инфляции.

Два правила, которые из этого следуют:
* номинальный показатель относят к **номинальному** ВВП, реальный — только к реальному (или вообще не относят);
* номинальный ряд превращается в реальный делением на индекс цен (`tf.deflate`): дефлятор ВВП для ВВП и его частей,
  ИПЦ — для доходов населения, выручки розницы, доходов бюджета (как приближение).

Натуральные показатели (м³, тонны) реальны по определению. Доли и ставки (`%`, `rate`) — ни то, ни другое: в отношении
цены сокращаются; исключение — реальная ставка = номинальная ставка − инфляционные ожидания (раздел 6.13).
""",
"en": r"""
### 3.5 Real vs nominal: the `real_nominal` field

A **nominal** indicator (`nominal`) is in current prices: GDP in roubles of that quarter, budget revenue, exports in dollars.
It grows both when the volume grows and when prices rise.
A **real** indicator (`real`) is in constant prices of a base year (real GDP "in 2021 prices") or in physical units
(m³ of gas, tonnes). Price growth is excluded from it.

The difference between them is **prices**:

$$\text{deflator}_t = 100 \cdot \frac{\text{nominal}_t}{\text{real}_t}, \qquad
(1+n) = (1+r)(1+p) \;\Rightarrow\; n = r + p + \frac{r\,p}{100}$$

where $n$ is nominal growth, $r$ real growth and $p$ price (deflator) growth. For small rates $n \approx r + p$;
the cross term matters only under high inflation.

Two rules follow:
* a nominal indicator is divided by **nominal** GDP, a real one only by real GDP (or not at all);
* a nominal series becomes real when divided by a price index (`tf.deflate`): the GDP deflator for GDP and its parts,
  the CPI for household income, retail sales, budget revenue (as an approximation).

Physical indicators (m³, tonnes) are real by definition. Shares and rates (`%`, `rate`) are neither: in a ratio prices
cancel out; the exception is the real rate = nominal rate − inflation expectations (section 6.13).
"""})

In [ ]:
tf.set_section('3.5', {'ru': 'Реальное и номинальное', 'en': 'Real vs nominal'})
gn, gr = SER[GDP_Q], SER[GDP_REAL_Q]
defl = tf.deflator(gn, gr)
base_year = (defl.groupby(defl.index.year).mean() - 100).abs().idxmin()
print(tr(f'Дефлятор ВВП = 100 · {GDP_Q} / {GDP_REAL_Q}: среднее за {base_year} г. = '
         f'{tf.ru_num(defl[str(base_year)].mean(), 1)} → реальный ВВП в ценах {base_year} г.',
         f'GDP deflator = 100 · {GDP_Q} / {GDP_REAL_Q}: the {base_year} average = '
         f'{tf.ru_num(defl[str(base_year)].mean(), 1)} → real GDP is in {base_year} prices'))

dq = tf.decompose_growth(gn, gr, 'Q')
da = tf.decompose_growth(gn, gr, 'Q', annual=True)
print(tr('Проверка тождества n = r + p + rp/100:', 'Check of the identity n = r + p + rp/100:'),
      np.allclose((dq['real'] + dq['prices'] + dq['cross']).dropna(), dq['nominal'].dropna()))
last = da.dropna().iloc[-1]
print(tr(f'Последние 4 квартала ({da.dropna().index[-1]}): номинальный ВВП {tf.pct(last["nominal"], 1)} г/г = '
         f'реальный {tf.pct(last["real"], 1)} + цены {tf.pct(last["prices"], 1)} + перекрёстный {tf.pct(last["cross"], 1)}',
         f'Last 4 quarters ({da.dropna().index[-1]}): nominal GDP {tf.pct(last["nominal"], 1)} y/y = '
         f'real {tf.pct(last["real"], 1)} + prices {tf.pct(last["prices"], 1)} + cross {tf.pct(last["cross"], 1)}'))
tf.plot_decomposition(da, tr('Номинальный рост ВВП = реальный рост + рост цен', 'Nominal GDP growth = real growth + price growth'),
                      tr('ВВП за 4 квартала к тем же 4 кварталам год назад, %; вклады, п.п.',
                         'GDP over 4 quarters vs the same 4 quarters a year earlier, %; contributions, p.p.'), start=START)
tf.plot_series({tr('дефлятор ВВП, г/г (4 кв.)', 'GDP deflator, y/y (4Q)'): da['prices'],
                tr('реальный ВВП, г/г (4 кв.)', 'real GDP, y/y (4Q)'): da['real']},
               tr('Цены и объём: две части номинального роста', 'Prices and volume: the two parts of nominal growth'), '%',
               start=START, hline=0)

In [ ]:
show_md({
"ru": r"""
Тот же приём для ряда, у которого нет «своего» реального двойника: доходы бюджета номинальны, дефлятируем их индексом
потребительских цен (цепной индекс из ИПЦ м/м). Получаем доходы **в ценах выбранного месяца** `REAL_PRICES_OF`:
по умолчанию — последнего месяца, за который есть ИПЦ. Можно указать любой месяц (`'2021-12'`) или год (`'2021'` —
в средних ценах 2021 г.). Смена базы сдвигает весь реальный ряд на один множитель, поэтому реальные темпы роста от неё
не зависят — меняется только «в каких рублях» мы смотрим на уровень.
""",
"en": r"""
The same trick for a series that has no real twin of its own: budget revenue is nominal, so we deflate it with the consumer
price index (a chain index built from CPI m/m). The result is revenue **in prices of the chosen month** `REAL_PRICES_OF`:
by default the last month with a CPI value. Any month (`'2021-12'`) or year (`'2021'` — average 2021 prices) can be set.
Changing the base shifts the whole real series by one factor, so real growth rates do not depend on it — only "in which
roubles" we look at the level changes.
"""})

In [ ]:
REAL_PRICES_OF = None        # None -- prices of the last month | цены последнего месяца; or/или '2021-12', '2021'

tb, tc = EX.get('cur_flow_cmlt'), EX.get('pop_growth')
if tb and tc:
    rev = tf.to_units(tf.decumulate(SER[tb]), C(tb))                  # monthly revenue, RUB | доходы за месяц, руб.
    cpi = tf.chain(tf.growth_x(SER[tc]))                               # CPI chain index | ИПЦ, цепной индекс
    base = REAL_PRICES_OF or str(cpi.dropna().index[-1])
    try:
        ok = pd.Series(cpi[base]).notna().any()
    except KeyError:
        ok = False
    if not ok:
        raise ValueError(tr(f'REAL_PRICES_OF = {base!r}: нет ИПЦ за этот период '
                            f'(есть с {cpi.dropna().index[0]} по {cpi.dropna().index[-1]})',
                            f'REAL_PRICES_OF = {base!r}: no CPI for this period '
                            f'(available from {cpi.dropna().index[0]} to {cpi.dropna().index[-1]})'))
    rev_real = tf.deflate(rev, cpi, base=base)
    n12, r12 = tf.rolling(rev, 'M', 'Y', 'sum'), tf.rolling(rev_real, 'M', 'Y', 'sum')
    tf.plot_series({tr('номинальные', 'nominal'): n12 / 1e12, tr(f'реальные, в ценах {base}', f'real, in {base} prices'): r12 / 1e12},
                   tr(f'{N(tb)}: номинальные и реальные', f'{N(tb)}: nominal and real'),
                   tr('сумма за 12 месяцев, трлн руб.; дефлятор — ИПЦ', '12-month sum, RUB tn; deflator — CPI'), start=START)
    tf.plot_series({tr('номинальный рост, г/г', 'nominal growth, y/y'): tf.yoy(n12, 'M'),
                    tr('реальный рост, г/г', 'real growth, y/y'): tf.yoy(r12, 'M')},
                   tr('Реальный рост доходов = номинальный за вычетом инфляции', 'Real revenue growth = nominal growth net of inflation'),
                   tr('сумма за 12 месяцев, % г/г', '12-month sum, % y/y'), start=START, hline=0)

In [ ]:
show_md({
"ru": r"""
### 3.6 Валюты

Правило выбора курса — из таблицы 3.2: поток — средний курс периода, запас — курс на конец периода (`tf.convert`, `how`).
Курс приводится к частоте ряда той же `tf.aggregate`. Отдельный случай — **накопленный поток (`cmlt`)**: делить накопленную
с начала года сумму на средний курс последнего месяца нельзя — январские рубли получены по январскому курсу. Правильно —
привести к месячным, перевести каждый месяц по его среднему курсу, снова накопить (`tf.convert_by_chars` делает это сам).
Пример с цифрами — в разделе 6.

### 3.7 Отношение к ВВП

ВВП — квартальный поток в рублях. Индикатор переводим в абсолютные единицы и в рубли, приводим к кварталу (поток —
суммой, запас — значением на конец), делим (`tf.pct_gdp`). Для потока два варианта: **квартал к кварталу ВВП** (`'q'`,
сильно сезонный) и **4 квартала к 4 кварталам** (`'4q'`, годовой показатель, обновляемый каждый квартал). Скользящая
*средняя* за 4 квартала к скользящей средней ВВП даёт ровно то же число — множитель 1/4 сокращается. Для запаса имеет
смысл только годовой знаменатель. ВВП выходит с задержкой около двух месяцев, поэтому последние точки %ВВП пусты.
""",
"en": r"""
### 3.6 Currencies

The rate rule comes from table 3.2: a flow uses the period-average rate, a stock the end-of-period rate (`tf.convert`, `how`).
The rate is brought to the series frequency with the same `tf.aggregate`. A special case is a **cumulative flow (`cmlt`)**:
the year-to-date sum must not be divided by the last month's average rate — January's roubles came in at January's rate.
The right way is to decumulate to monthly values, convert each month at its own average rate and cumulate again
(`tf.convert_by_chars` does this itself). A numerical example is in section 6.

### 3.7 Ratio to GDP

GDP is a quarterly flow in roubles. The indicator is converted to absolute units and roubles, brought to quarters (a flow by
summing, a stock by the end value) and divided (`tf.pct_gdp`). For a flow there are two options: **quarter to quarterly GDP**
(`'q'`, strongly seasonal) and **4 quarters to 4 quarters** (`'4q'`, an annual measure updated every quarter). The rolling
4-quarter *average* over the rolling GDP average gives exactly the same number — the 1/4 factor cancels. For a stock only an
annual denominator makes sense. GDP comes out about two months late, so the last % of GDP points are empty.
"""})

In [ ]:
g4 = tf.rolling(GDP_UNITS, 'Q', 'Y', 'sum')
print(tr('Сумма за 4 кв. / сумма ВВП = среднее за 4 кв. / среднее ВВП:', '4Q sum / GDP sum = 4Q average / GDP average:'),
      tf.same(tf.rolling(GDP_UNITS, 'Q', 'Y', 'mean') * 4, g4))

In [ ]:
show_md({
"ru": r"""
## 4. Карта преобразований: что из чего получается

Ключевая идея: **все преобразования идут через уровень**. Вопрос «можно ли получить X из Y» сводится к вопросу
«можно ли из Y восстановить уровень и насколько точно».

* `none` — уровень есть. Можно всё в пределах типа ряда (запасы не накапливают, потоки не сравнивают с декабрём).
* `cmlt` — уровень восстанавливается **точно** разностью внутри года.
* `mom`/`qoq`/`wow` и `ytd` — уровень восстанавливается **с точностью до множителя** (цепной индекс). Всё, что не зависит
  от масштаба (любые приросты, cmlt/cmlt), — можно; всё, что требует единиц (рубли, %ВВП, валюта), — нельзя без
  хотя бы одного уровня-«якоря». `ytd` и `mom` несут одну и ту же информацию.
* `yoy` и `cmlt_cmlt` — уровень **не восстанавливается**: каждое значение связывает период только с тем же периодом
  год назад. Нужен целый базовый год уровней.
* `%`, `rate`, `%gdp` — уже относительные величины: изменения в п.п., средние.

`tf.available(характеристики)` превращает правила в таблицу: ✓ — можно; ✓* — только с дополнительными данными
(якорь, базовый год, индекс цен); ◇ — посчитать можно, но смысла нет (ставка г/г в процентах, сумма запасов, кубометры
к ВВП в рублях); ✗ — не применимо (например, агрегация годового ряда в год).
**Это подсказка, а не запрет**: функции модуля посчитают что угодно, ответственность за смысл — на аналитике.
""",
"en": r"""
## 4. The transformation map: what can be derived from what

The key idea: **every transformation goes through the level**. The question "can X be obtained from Y" reduces to
"can the level be recovered from Y, and how precisely".

* `none` — the level is there. Everything is possible within the series type (stocks are not cumulated, flows are not compared with December).
* `cmlt` — the level is recovered **exactly** by differencing within the year.
* `mom`/`qoq`/`wow` and `ytd` — the level is recovered **up to a factor** (a chain index). Everything scale-free (any growth
  rates, cmlt/cmlt) is possible; everything that needs units (roubles, % of GDP, currency) is not, without at least one
  "anchor" level. `ytd` and `mom` carry the same information.
* `yoy` and `cmlt_cmlt` — the level **cannot be recovered**: each value links a period only to the same period a year
  earlier. A whole base year of levels is needed.
* `%`, `rate`, `%gdp` — already relative quantities: changes in p.p., averages.

`tf.available(characteristics)` turns the rules into a table: ✓ — possible; ✓* — only with extra data (an anchor, a base year,
a price index); ◇ — computable but meaningless (a rate y/y in percent, a sum of stocks, cubic metres over GDP in roubles);
✗ — not applicable (e.g. aggregating an annual series to a year).
**This is a hint, not a ban**: the module's functions compute anything; responsibility for the meaning is the analyst's.
"""})

In [ ]:
CLASSES = [
    (('валюта, поток, номинальный', 'money, flow, nominal'), dict(metric='index', transf='none', stock_flow='flow', dim='currency', currency='RUB', real_nominal='nominal')),
    (('валюта, поток, реальный', 'money, flow, real'), dict(metric='index', transf='none', stock_flow='flow', dim='currency', currency='RUB', real_nominal='real')),
    (('валюта, запас', 'money, stock'), dict(metric='index', transf='none', stock_flow='stock', dim='currency', currency='USD', real_nominal='nominal')),
    (('натуральные единицы, поток', 'physical units, flow'), dict(metric='index', transf='none', stock_flow='flow', dim='m3', currency='none', real_nominal='real')),
    (('валюта, поток, с начала года', 'money, flow, year-to-date'), dict(metric='index', transf='cmlt', stock_flow='flow', dim='currency', currency='RUB', real_nominal='nominal')),
    (('к пред. периоду (mom/qoq/wow)', 'vs previous period (mom/qoq/wow)'), dict(metric='index', transf='mom', stock_flow='none', dim='none', currency='none', real_nominal='none')),
    (('к концу прошлого года (ytd)', 'vs end of last year (ytd)'), dict(metric='index', transf='ytd', stock_flow='stock', dim='none', currency='none', real_nominal='nominal')),
    (('г/г (yoy)', 'y/y (yoy)'), dict(metric='index', transf='yoy', stock_flow='flow', dim='none', currency='none', real_nominal='real')),
    (('с начала года к прошлому году', 'year-to-date vs last year'), dict(metric='index', transf='cmlt_cmlt', stock_flow='flow', dim='none', currency='none', real_nominal='real')),
    (('доля, %', 'share, %'), dict(metric='%', transf='none', stock_flow='none', dim='none', currency='none')),
    (('ставка, % годовых', 'rate, % p.a.'), dict(metric='rate', transf='none', stock_flow='none', dim='none', currency='none')),
]
matrix = pd.DataFrame({tr(*nm): {lab: tf.available(dict(cp, freq='M', eop_avg='none', ticker=nm[1]))[k][0]
                                 for k, lab in tf.targets()} for nm, cp in CLASSES}).T
display(matrix)
print(tr('Пояснения к любой строке: tf.avail_table(характеристики), например:',
         'Notes for any row: tf.avail_table(characteristics), for example:'))
display(tf.avail_table(dict(CLASSES[5][1], freq='M', eop_avg='avg', ticker='')))

In [ ]:
show_md({
"ru": r"""
### 4.2 Доказательства на реальном ряду

**Г/г не определяет уровень.** Возьмём настоящий ряд A — добычу газа — и умножим каждый январь на одно и то же число,
каждый февраль — на другое и т.д. (B = A × сезонный множитель). Это не новые данные, а контрпример: г/г у A и B
совпадают **точно** — множитель сокращается в отношении «март к марту», — а помесячная динамика совершенно разная.
По г/г невозможно понять, какой из двух рядов «настоящий», значит, м/м, накопленные и уровень из г/г не восстановить.
""",
"en": r"""
### 4.2 Proofs on a real series

**Y/y does not pin down the level.** Take a real series A — gas production — and multiply every January by one number,
every February by another and so on (B = A × a seasonal factor). These are not new data but a counterexample: the y/y of A
and B coincide **exactly** — the factor cancels in "March over March" — while the month-to-month dynamics are completely
different. From y/y one cannot tell which series is "real", so m/m, cumulative values and the level cannot be recovered from y/y.
"""})

In [ ]:
tf.set_section('4.2', {'ru': 'Что нельзя получить из г/г и м/м', 'en': 'What cannot be derived from y/y and m/m'})
tg = EX.get('nat_flow')
A = SER[tg].dropna()['2018':'2024'] if tg else None
if A is not None:
    B = A * np.tile([1.15, 0.90, 1.00, 1.05, 0.95, 1.00, 1.10, 0.85, 1.00, 1.00, 0.95, 1.05], len(A) // 12 + 1)[:len(A)]
    print(tr('г/г совпадают:', 'y/y coincide:'), tf.same(tf.yoy(A, 'M'), tf.yoy(B, 'M')),
          tr('| м/м совпадают:', '| m/m coincide:'), tf.same(tf.pop(A), tf.pop(B)))
    tf.plot_series({tr('A: добыча газа', 'A: gas production'): A, tr('B = A × сезонный множитель', 'B = A × seasonal factor'): B},
                   tr('Два разных ряда…', 'Two different series…'), tr('млн м³ в месяц', 'mn m³ per month'))
    tf.plot_series({tr('г/г A', 'y/y A'): tf.yoy(A, 'M'), tr('г/г B', 'y/y B'): tf.yoy(B, 'M')},
                   tr('…с одинаковыми г/г', '…with identical y/y'), tr('линии совпадают, %', 'the lines coincide, %'),
                   dashed=[tr('г/г B', 'y/y B')], hline=0)

In [ ]:
show_md({
"ru": r"""
**М/м определяют уровень только с точностью до множителя.** У рядов A и 3A одинаковы м/м, г/г, cmlt/cmlt — любые
относительные показатели. Но годовой объём различается втрое. Поэтому из м/м строится цепной индекс (база 100 —
в произвольный момент), но не единицы и не %ВВП. Если известен **хотя бы один** уровень, множитель определён.

**Г/г и cmlt/cmlt + один базовый год = полный ряд:** $L_t = L_{t-12}(1+g_t)$; для cmlt/cmlt сначала накопленные
$C_t = C_{t-12}(1+g_t)$, потом приведение к месячным.
""",
"en": r"""
**M/m pins down the level only up to a factor.** Series A and 3A have the same m/m, y/y, cmlt/cmlt — any relative
measure. But their annual volumes differ threefold. So m/m gives a chain index (base 100 at an arbitrary point), but not
units and not % of GDP. Once **at least one** level is known, the factor is fixed.

**Y/y and cmlt/cmlt + one base year = the full series:** $L_t = L_{t-12}(1+g_t)$; for cmlt/cmlt first the cumulative
values $C_t = C_{t-12}(1+g_t)$, then decumulation to monthly values.
"""})

In [ ]:
if A is not None:
    print(tr('м/м A и 3A совпадают:', 'm/m of A and 3A coincide:'), tf.same(tf.pop(A), tf.pop(3 * A)),
          tr('| cmlt/cmlt совпадают:', '| cmlt/cmlt coincide:'), tf.same(tf.cmlt_cmlt(A, 'M'), tf.cmlt_cmlt(3 * A, 'M')))
    m3 = tr('м³', 'm³')
    print(tr('добыча за 2024 г.: A =', '2024 production: A ='), tf.fmt_amount(A['2024'].sum() * 1e6, m3),
          '| 3A =', tf.fmt_amount(3 * A['2024'].sum() * 1e6, m3))
    rec_y = tf.level_from_yoy(tf.yoy(A, 'M')['2019':], A['2018'], 'M')
    rec_cc = tf.level_from_cmlt_cmlt(tf.cmlt_cmlt(A, 'M')['2019':], A['2018'], 'M')
    print(tr('уровень из г/г + 2018 год восстановлен точно:', 'level from y/y + year 2018 recovered exactly:'), tf.same(rec_y, A),
          tr('| из cmlt/cmlt + 2018 год:', '| from cmlt/cmlt + year 2018:'), tf.same(rec_cc, A))

In [ ]:
show_md({
"ru": r"""
### 4.3 Что значит конкретная цифра: `tf.explain`

Характеристики + последнее значение → обычная фраза. То, что аналитик проговаривает про себя, увидев новую публикацию.
Дальше `explain` открывает каждый пример.
""",
"en": r"""
### 4.3 What a specific number means: `tf.explain`

Characteristics + the latest value → a plain sentence. What an analyst says to themselves on seeing a new release.
Below, `explain` opens every example.
"""})

In [ ]:
show_md({
"ru": r"""
## 5. Один график — все метрики одного индикатора

Берём уровень одного показателя и на одной фигуре считаем из него всё стандартное: приросты, накопленные,
скользящие суммы, агрегацию, индекс, пересчёт в другую валюту, %ВВП. Под заголовком каждой панели — формула.
`tf.all_metrics` считает, `tf.plot_all_metrics` рисует; на вход — ряд **уровней в абсолютных единицах**,
его частота, тип (поток / запас / среднее) и, если это деньги, валюта.

Для потока — экспорт товаров по платёжному балансу (квартальный поток в долларах, переводим в рубли и относим к ВВП):
""",
"en": r"""
## 5. One figure — every metric of one indicator

Take the level of one indicator and compute everything standard from it on one figure: growth rates, cumulative values,
rolling sums, aggregation, an index, conversion to another currency, % of GDP. The formula is under each panel's title.
`tf.all_metrics` computes, `tf.plot_all_metrics` draws; the input is a series of **levels in absolute units**, its
frequency, its type (flow / stock / average) and, if it is money, its currency.

For a flow — balance-of-payments exports of goods (a quarterly flow in dollars, converted to roubles and divided by GDP):
"""})

In [ ]:
tf.set_section('5', {'ru': 'Все метрики одного индикатора', 'en': 'Every metric of one indicator'})
t = EX.get('fcur_flow')
if t:
    c = C(t)
    lvl = tf.to_units(SER[t], c)
    M_ = tf.all_metrics(lvl, c['freq'], tf.kind_of(c), ccy=c['currency'], base_ccy=BASE_CCY,
                        fx_target=BASE_CCY, gdp_q_units=GDP_UNITS, base_period='2021')
    tf.plot_all_metrics(M_, tr(f'{N(t)}: из одного уровня — все метрики', f'{N(t)}: every metric from one level'),
                        tr(f'{t}; в рубли — по среднему курсу квартала', f'{t}; to roubles at the quarterly average rate'),
                        scale=1e9, scale_txt=tr('млрд долл.', 'USD bn'), start='2015')

In [ ]:
show_md({
"ru": r"""
Для запаса набор другой: нет накопленных и cmlt/cmlt (снимки не складываются), вместо суммы за год — средний запас,
появляется прирост к концу прошлого года, курс — на конец периода. Международные резервы:
""",
"en": r"""
For a stock the set is different: no cumulative values and no cmlt/cmlt (snapshots are not summed), the average stock instead
of the annual sum, growth vs the end of last year appears, and the rate is end-of-period. International reserves:
"""})

In [ ]:
t = EX.get('fcur_stock')
if t:
    c = C(t)
    lvl = tf.to_units(SER[t], c)
    M_ = tf.all_metrics(lvl, c['freq'], tf.kind_of(c), ccy=c['currency'], base_ccy=BASE_CCY,
                        fx_target=BASE_CCY, gdp_q_units=GDP_UNITS, base_period='2021')
    tf.plot_all_metrics(M_, tr(f'{N(t)}: метрики запаса', f'{N(t)}: stock metrics'),
                        tr(f'{t}; в рубли — по курсу на конец месяца', f'{t}; to roubles at the end-of-month rate'),
                        scale=1e9, scale_txt=tr('млрд долл.', 'USD bn'), start='2015')

In [ ]:
show_md({
"ru": r"""
## 6. Живые примеры по типам показателей

Каждый пример: что публикует ведомство → что значит последняя цифра (`tf.explain`) → что из неё можно сделать → расчёт.
Если пример пропущен — среди описанных рядов нет показателя такого типа; укажите тикер в `TYPE_OVERRIDES`.
""",
"en": r"""
## 6. Live examples by indicator type

Each example: what the agency publishes → what the latest number means (`tf.explain`) → what can be done with it → the calculation.
If an example is skipped, none of the described series has that type; set a ticker in `TYPE_OVERRIDES`.
"""})

In [ ]:
def need(key):
    """Ticker for an indicator type, or a message why the example is skipped.
    Тикер для типа показателя или сообщение, почему пример пропущен."""
    t = EX.get(key)
    if t is None:
        print(tr(f'Пример «{tf.type_title(key)}» пропущен: укажите тикер в TYPE_OVERRIDES["{key}"]',
                 f'Example "{tf.type_title(key)}" skipped: set a ticker in TYPE_OVERRIDES["{key}"]'))
    return t

In [ ]:
show_md({
"ru": r"""
### 6.1 Показатель в валюте, поток, номинальный

Росстат публикует номинальный ВВП за квартал в млрд руб. Поток: годовой ВВП — сумма четырёх кварталов, «ВВП за последние
12 месяцев» — скользящая сумма четырёх кварталов (знаменатель для годовых отношений к ВВП). В доллары — по **среднему**
курсу каждого квартала, потом суммирование.
""",
"en": r"""
### 6.1 Money indicator, flow, nominal

Rosstat publishes nominal GDP per quarter in RUB bn. It is a flow: annual GDP is the sum of four quarters, "GDP over the
last 12 months" is the rolling sum of four quarters (the denominator for annual ratios to GDP). To dollars — at the
**average** rate of each quarter, then sum.
"""})

In [ ]:
tf.set_section('6.1', {'ru': 'Валюта, поток, номинальный', 'en': 'Money, flow, nominal'})
t = need('cur_flow_nominal')
if t:
    c = C(t)
    tf.explain(c, SER[t])
    u = tf.to_units(SER[t], c)
    u4 = tf.rolling(u, 'Q', 'Y', 'sum')
    usd_q = tf.convert(u, c['currency'], 'USD', 'Q', 'mean')
    usd_4 = tf.rolling(usd_q, 'Q', 'Y', 'sum')
    alt = u4 / tf.rolling(tf.fx_at(c['currency'], u.index, 'Q', 'mean'), 'Q', 'Y', 'mean')
    gap = 100 * (alt / usd_4 - 1)
    print(tr(f'В долларах: «каждый квартал по своему курсу» и «годовая сумма / средний курс за год» расходятся до '
             f'{tf.ru_num(gap.abs().max())}% (в {gap.abs().idxmax()})',
             f'In dollars: "each quarter at its own rate" and "annual sum / annual average rate" differ by up to '
             f'{tf.ru_num(gap.abs().max())}% (in {gap.abs().idxmax()})'))
    q_, s4 = tr('за квартал', 'quarter'), tr('сумма за 4 квартала', 'sum over 4 quarters')
    tf.plot_series({q_: u / 1e12, s4: u4 / 1e12}, N(t), tr('трлн руб.', 'RUB tn'), bars=[q_], start=START)
    tf.plot_series({s4: usd_4 / 1e9}, tr(f'{N(t)} в долларах', f'{N(t)} in dollars'),
                   tr('млрд долл.; каждый квартал — по среднему курсу квартала', 'USD bn; each quarter at its average rate'),
                   start=START, nd=0)

In [ ]:
show_md({
"ru": r"""
### 6.2 Показатель в валюте, поток, реальный

Реальный ВВП — в постоянных ценах базового года. Его рост — «физический объём»; к номинальному ВВП и валюте его
не относят (в долларах «в ценах 2021 г.» смысла нет). Темпы роста — квартал к кварталу прошлого года и за 4 квартала.
""",
"en": r"""
### 6.2 Money indicator, flow, real

Real GDP is in constant prices of a base year. Its growth is the "physical volume"; it is not divided by nominal GDP or
converted to a currency (dollars "in 2021 prices" make no sense). Growth rates — quarter vs the same quarter last year and over 4 quarters.
"""})

In [ ]:
tf.set_section('6.2', {'ru': 'Валюта, поток, реальный', 'en': 'Money, flow, real'})
t = need('cur_flow_real')
if t:
    c = C(t)
    tf.explain(c, SER[t])
    u = tf.to_units(SER[t], c)
    qq = tr('квартал к кварталу год назад', 'quarter vs same quarter a year ago')
    tf.plot_series({qq: tf.yoy(u, 'Q'), tr('4 квартала к 4 кварталам', '4 quarters vs 4 quarters'): tf.yoy(tf.rolling(u, 'Q', 'Y', 'sum'), 'Q')},
                   tr(f'{N(t)}: рост', f'{N(t)}: growth'), tr('%, г/г', '%, y/y'), bars=[qq], start=START, hline=0)

In [ ]:
show_md({
"ru": r"""
### 6.3 Показатель в валюте, поток, накопленным итогом с начала года

Казначейство публикует исполнение бюджета **нарастающим итогом**: цифра за март — это январь–март. Отсюда:
месяц — разность соседних накопленных (приводим к месячным); годовой объём — прямо из накопленных $C_t + C_\text{дек} - C_{t-12}$;
квартал — значения на конец кварталов и их разность; в доллары — только помесячно.
""",
"en": r"""
### 6.3 Money indicator, flow, cumulative since January

The Treasury publishes budget execution **cumulatively**: the March figure is January–March. Hence:
a month is the difference of neighbouring cumulative values (decumulation to monthly values); the annual volume comes
directly from cumulative values $C_t + C_\text{Dec} - C_{t-12}$; a quarter is the difference of quarter-end values;
to dollars — only month by month.
"""})

In [ ]:
tf.set_section('6.3', {'ru': 'Валюта, поток, с начала года', 'en': 'Money, flow, year-to-date'})
t = need('cur_flow_cmlt')
if t:
    c = C(t)
    tf.explain(c, SER[t])
    cu = tf.to_units(SER[t], c)
    mu = tf.decumulate(cu)
    lumped = tf.lumped_periods(SER[t], 'M')
    if lumped:
        print(tr('После приведения к месячным значения за', 'After decumulation the values for'), [str(x) for x in lumped[:5]],
              tr('включают несколько месяцев.', 'cover several months.'))
    cm_, mm_ = tr('накопленным итогом', 'cumulative'), tr('за месяц', 'month')
    tf.plot_series({cm_: cu / 1e12, mm_: mu / 1e12}, N(t), tr('трлн руб.', 'RUB tn'), bars=[mm_], start='2021')
    qu = tf.decumulate(tf.aggregate(cu, 'Q', 'last', 'M'))                    # quarterly flow from cumulative | квартальный поток
    qg = tr('за квартал к ВВП квартала', 'quarter to quarterly GDP')
    tf.plot_series({qg: tf.pct_gdp(qu, GDP_UNITS, 'Q', 'flow', 'q'),
                    tr('за 4 квартала к ВВП за 4 квартала', '4 quarters to 4-quarter GDP'): tf.pct_gdp(qu, GDP_UNITS, 'Q', 'flow', '4q')},
                   N(t), tr('% ВВП', '% of GDP'), bars=[qg], start=START)
    right = tf.convert_by_chars(cu, c, 'USD')
    wrong = tf.convert(cu, c['currency'], 'USD', 'M', 'mean')
    dec = right.index.month == 12
    cmp_ = pd.DataFrame({tr('верно: помесячно по курсу месяца', 'right: month by month at its rate'): right[dec] / 1e9,
                         tr('неверно: декабрьское накопленное / курс декабря', 'wrong: December cumulative / December rate'): wrong[dec] / 1e9})
    cmp_[tr('ошибка, %', 'error, %')] = 100 * (cmp_.iloc[:, 1] / cmp_.iloc[:, 0] - 1)
    cmp_.index = cmp_.index.year
    print(tr('Годовые доходы, млрд долл.:', 'Annual revenue, USD bn:'))
    display(cmp_.tail(8).round(1))

In [ ]:
show_md({
"ru": r"""
### 6.4 Показатель в иностранной валюте, поток

Банк России публикует платёжный баланс в **миллионах** долларов (`mult = 10⁶`), не в миллиардах. Для доли в ВВП:
× mult → в рубли по **среднему** курсу квартала → к ВВП квартала или 4 кв. к 4 кв.
""",
"en": r"""
### 6.4 Foreign-currency indicator, flow

The Bank of Russia publishes the balance of payments in **millions** of dollars (`mult = 10⁶`), not billions. For the
share of GDP: × mult → to roubles at the quarter's **average** rate → to quarterly GDP or 4Q to 4Q.
"""})

In [ ]:
tf.set_section('6.4', {'ru': 'Иностранная валюта, поток', 'en': 'Foreign currency, flow'})
t = need('fcur_flow')
if t:
    c = C(t)
    f = c['freq']
    tf.explain(c, SER[t])
    u = tf.to_units(SER[t], c)
    rub = tf.to_units(tf.convert_by_chars(SER[t], c, BASE_CCY), c)
    pp_, qg = tr('за период', 'period'), tr('квартал к ВВП квартала', 'quarter to quarterly GDP')
    tf.plot_series({pp_: u / 1e9, tr('сумма за год', 'annual sum'): tf.rolling(u, f, 'Y', 'sum') / 1e9},
                   N(t), tr(f'млрд {tf.unit_label(c)}', f'{tf.unit_label(c)} bn'), bars=[pp_], start=START, nd=0)
    tf.plot_series({qg: tf.pct_gdp(rub, GDP_UNITS, f, 'flow', 'q'),
                    tr('4 квартала к ВВП за 4 квартала', '4 quarters to 4-quarter GDP'): tf.pct_gdp(rub, GDP_UNITS, f, 'flow', '4q')},
                   N(t), tr('% ВВП; пересчёт по среднему курсу квартала', '% of GDP; converted at the quarterly average rate'),
                   bars=[qg], start=START)

In [ ]:
show_md({
"ru": r"""
### 6.5 Показатель в иностранной валюте, запас

Резервы — снимок на дату. Сумма месячных значений бессмысленна; в квартал — значение на конец, в рубли — по курсу на
ту же дату. Знаменатель для доли в ВВП — ВВП за 4 квартала.
""",
"en": r"""
### 6.5 Foreign-currency indicator, stock

Reserves are a snapshot at a date. A sum of monthly values is meaningless; to a quarter — the end value, to roubles — at
the rate of the same date. The denominator for the share of GDP is GDP over 4 quarters.
"""})

In [ ]:
tf.set_section('6.5', {'ru': 'Иностранная валюта, запас', 'en': 'Foreign currency, stock'})
t = need('fcur_stock')
if t:
    c = C(t)
    f = c['freq']
    tf.explain(c, SER[t])
    u = tf.to_units(SER[t], c)
    um = u if f == 'M' else tf.aggregate(u, 'M', 'last', f)
    eop = tf.convert(um, c['currency'], BASE_CCY, 'M', 'last')
    avg = tf.convert(um, c['currency'], BASE_CCY, 'M', 'mean')
    wr = tr('по среднему курсу месяца (неверно для запаса)', 'at the monthly average rate (wrong for a stock)')
    tf.plot_series({tr('по курсу на конец месяца (верно)', 'at the end-of-month rate (right)'): eop / 1e12, wr: avg / 1e12},
                   tr(f'{N(t)} в рублях', f'{N(t)} in roubles'), tr('трлн руб.', 'RUB tn'), start='2021', dashed=[wr])
    wq = tr('на конец квартала к ВВП квартала (неверно)', 'end of quarter to quarterly GDP (wrong)')
    tf.plot_series({tr('на конец квартала к ВВП за 4 квартала', 'end of quarter to 4-quarter GDP'): tf.pct_gdp(eop, GDP_UNITS, 'M', 'stock', '4q'),
                    wq: tf.pct_gdp(eop, GDP_UNITS, 'M', 'stock', 'q')},
                   N(t), tr('% ВВП: запас к квартальному ВВП завышен в 4 раза и сезонен',
                            '% of GDP: a stock over quarterly GDP is 4 times too high and seasonal'), start=START, dashed=[wq])

In [ ]:
show_md({
"ru": r"""
### 6.6 Показатель в натуральных единицах, поток

Добыча газа в м³: всё, что касается времени, — можно (суммы за год, приросты, сезонность). Перевести в валюту или
отнести к ВВП — посчитать можно, но смысла нет: кубометры не делятся на рубли. Натуральный показатель реален по определению.
""",
"en": r"""
### 6.6 Physical-unit indicator, flow

Gas production in m³: everything about time is possible (annual sums, growth rates, seasonality). Converting it to a
currency or dividing by GDP is computable but meaningless: cubic metres do not divide by roubles. A physical indicator is
real by definition.
"""})

In [ ]:
tf.set_section('6.6', {'ru': 'Натуральные единицы, поток', 'en': 'Physical units, flow'})
t = need('nat_flow')
if t:
    c = C(t)
    f = c['freq']
    tf.explain(c, SER[t])
    u = tf.to_units(SER[t], c)
    s12 = tf.rolling(u, f, 'Y', 'sum')
    ub = tr(f'млрд {tf.unit_label(c)}', f'bn {tf.unit_label(c)}')
    mo, sm = tr('за месяц', 'month'), tr('сумма за 12 мес.', '12-month sum')
    tf.plot_series({mo: u / 1e9, sm: s12 / 1e9}, N(t), ub, bars=[mo], right=[sm], start=START)
    tf.plot_series({tr('г/г месяца', 'y/y of the month'): tf.yoy(u, f), tr('г/г суммы за 12 мес.', 'y/y of the 12-month sum'): tf.yoy(s12, f)},
                   tr(f'{N(t)}: у суммы за 12 мес. шума меньше', f'{N(t)}: the 12-month sum is less noisy'), tr('%, г/г', '%, y/y'),
                   start=START, hline=0)
    tf.season_chart(u / 1e9, f, tr(f'{N(t)} по месяцам', f'{N(t)} by month'), tr(f'{ub}; каждая линия — год', f'{ub}; one line per year'))
    tf.plot_year_range(u / 1e9, f, tr(f'{N(t)}: последние два года на фоне прошлых лет', f'{N(t)}: the last two years against past years'),
                       tr(f'{ub}; область — мин–макс по каждому месяцу за более ранние годы',
                          f'{ub}; band — min–max for each month over earlier years'))

In [ ]:
show_md({
"ru": r"""
### 6.7 Прирост к предыдущему месяцу

ИПЦ м/м. Из м/м строится цепной индекс, а из него всё относительное: г/г, к декабрю, квартальные приросты среднего индекса.
В рубли и %ВВП — нельзя: у прироста нет единиц. Аннуализировать м/м без сезонной корректировки — тоже нельзя.
""",
"en": r"""
### 6.7 Growth vs previous month

CPI m/m. A chain index is built from m/m, and from it everything relative: y/y, vs December, quarterly growth of the
average index. Roubles and % of GDP are not possible: growth has no units. Annualising m/m without seasonal adjustment is wrong too.
"""})

In [ ]:
tf.set_section('6.7', {'ru': 'Прирост к предыдущему месяцу', 'en': 'Growth vs previous month'})
t = need('pop_growth')
if t:
    c = C(t)
    f = c['freq']
    tf.explain(c, SER[t])
    x = tf.growth_x(SER[t])
    I = tf.chain(x)
    Iq = tf.aggregate(I, 'Q', 'mean', f)                     # prices -- period average | цены -- среднее за период
    print(tr(f'Последний полный квартал {Iq.index[-1]}: к/к {tf.pct(tf.pop(Iq).iloc[-1])}, г/г {tf.pct(tf.yoy(Iq, "Q").iloc[-1])}',
             f'Last full quarter {Iq.index[-1]}: q/q {tf.pct(tf.pop(Iq).iloc[-1])}, y/y {tf.pct(tf.yoy(Iq, "Q").iloc[-1])}'))
    mm, yy = tr('м/м', 'm/m'), tr('г/г', 'y/y')
    tf.plot_series({mm: x, yy: tf.yoy(I, f)}, N(t), '%', bars=[mm], right=[yy], start='2021', nd=2)
    tf.plot_series({tr('к декабрю прошлого года', 'vs December of last year'): tf.ytd(I, f)},
                   tr(f'{N(t)}: с начала года', f'{N(t)}: year to date'), '%', start='2021', hline=0, nd=2)
    tf.season_chart(x, f, tr(f'{N(t)}: м/м по месяцам', f'{N(t)}: m/m by month'),
                    tr('%; сезонность видна — аннуализировать нельзя', '%; seasonality is visible — do not annualise'), nd=2)
    tf.plot_year_range(x, f, tr(f'{N(t)}: последние два года на фоне прошлых лет', f'{N(t)}: the last two years against past years'),
                       tr('%, м/м; область — мин–макс по каждому месяцу; 2014–2015 и 2022 исключены как шоковые',
                          '%, m/m; band — min–max for each month; 2014–2015 and 2022 excluded as shock years'),
                       exclude=(2014, 2015, 2022), hline=0, nd=2)

In [ ]:
show_md({
"ru": r"""
### 6.8 Прирост к предыдущей неделе

Недельная инфляция — цепные приросты по неделям. Недели не совпадают с месяцами и годом, поэтому «месячные» и
«годовые» приросты из недельных — приближение; ценность недельных — в оперативности.

Насколько приближение точное, проверяем прямо: восстанавливаем м/м из недельных двумя способами (по индексу последней
недели месяца и по среднему недельному индексу за месяц), вычитаем официальный месячный м/м и считаем статистику
ошибки (`tf.compare_estimates`): смещение и его значимость, медиану, MAE, RMSE и RMSE относительно разброса самой инфляции.
""",
"en": r"""
### 6.8 Growth vs previous week

Weekly inflation is a chain of week-on-week changes. Weeks do not line up with months and years, so "monthly" and
"annual" growth from weekly data is an approximation; the value of weekly data is its timeliness.

We test the accuracy directly: rebuild m/m from weekly data in two ways (from the index of the last week of the month and
from the average weekly index over the month), subtract the official monthly m/m and compute error statistics
(`tf.compare_estimates`): bias and its significance, median, MAE, RMSE and RMSE relative to the spread of inflation itself.
"""})

In [ ]:
tf.set_section('6.8', {'ru': 'Прирост к предыдущей неделе', 'en': 'Growth vs previous week'})
t = need('wow_growth')
if t:
    c = C(t)
    tf.explain(c, SER[t])
    x = tf.growth_x(SER[t])
    I = tf.chain(x)
    Im = tf.aggregate(I, 'M', 'last', 'W')
    est_last = tf.pop(Im)                                         # last week of the month | по последней неделе месяца
    est_avg = tf.pop(tf.aggregate(I, 'M', 'mean', 'W'))           # average weekly index | по среднему недельному индексу
    if EX.get('pop_growth'):
        ref = tf.growth_x(SER[EX['pop_growth']])
        tf.plot_series({tr('«м/м» по последней неделе месяца', '"m/m" from the last week of the month'): est_last,
                        tr('м/м по месячным данным', 'm/m from monthly data'): ref},
                       tr('Недельные приросты дают приближённые месячные', 'Weekly data give approximate monthly growth'),
                       '%', start='2023', nd=2)
        k_last, k_avg = tr('по последней неделе', 'last week'), tr('по среднему за месяц', 'monthly average')
        res = {k_last: tf.compare_estimates(est_last[START:], ref[START:]),
               k_avg: tf.compare_estimates(est_avg[START:], ref[START:])}
        e1 = tr('ошибка: последняя неделя − месячные данные', 'error: last week − monthly data')
        tf.plot_series({e1: res[k_last]['error'],
                        tr('ошибка: среднее за месяц − месячные данные', 'error: monthly average − monthly data'): res[k_avg]['error']},
                       tr('Ошибка оценки м/м по недельным данным', 'Error of the m/m estimate from weekly data'),
                       tr('п.п.; положительная — недельные завышают', 'p.p.; positive — weekly data overstate'),
                       bars=[e1], start=START, hline=0, nd=2)
        rows = {name: {tr('наблюдений', 'observations'): r['n'],
                       tr('средняя ошибка (смещение), п.п.', 'mean error (bias), p.p.'): r['bias'],
                       tr('t-статистика смещения', 'bias t-statistic'): r['t_bias'],
                       tr('p-значение смещения', 'bias p-value'): r['p_bias'],
                       tr('медиана ошибки, п.п.', 'median error, p.p.'): r['median'],
                       tr('MAE, п.п.', 'MAE, p.p.'): r['mae'], tr('RMSE, п.п.', 'RMSE, p.p.'): r['rmse'],
                       tr('ст. откл. м/м инфляции, п.п.', 'std of m/m inflation, p.p.'): r['std_ref'],
                       tr('RMSE / ст. откл.', 'RMSE / std'): r['rmse_to_std'], tr('корреляция', 'correlation'): r['corr'],
                       tr('доля |ошибки| > 0,1 п.п.', 'share |error| > 0.1 p.p.'): r['share_gt_0.1'],
                       tr('доля |ошибки| > 0,2 п.п.', 'share |error| > 0.2 p.p.'): r['share_gt_0.2']}
                for name, r in res.items()}
        display(pd.DataFrame(rows).round(3))
        r = res[k_last]
        ref_s = ref[START:].dropna()
        if r['p_bias'] > 0.05:
            verdict = tr('систематического смещения нет (p > 0,05)', 'no systematic bias (p > 0.05)')
        else:
            verdict = tr(f'есть систематическое смещение {tf.ru_num(r["bias"], 2)} п.п. (p = {tf.ru_num(r["p_bias"], 3)})',
                         f'there is a systematic bias of {tf.ru_num(r["bias"], 2)} p.p. (p = {tf.ru_num(r["p_bias"], 3)})')
        show_md({
            'ru': f'**Насколько значима ошибка (с {START[:4]} г.).** Типичная ошибка (RMSE) — **{tf.ru_num(r["rmse"], 2)} п.п.**, '
                  f'медиана — {tf.ru_num(r["median"], 2)} п.п.; {verdict}. Для сравнения: м/м инфляция колеблется со стандартным '
                  f'отклонением {tf.ru_num(r["std_ref"], 2)} п.п., средний её уровень — {tf.ru_num(ref_s.mean(), 2)}%, '
                  f'т.е. ошибка составляет **{tf.ru_num(100 * r["rmse_to_std"], 0)}%** от разброса самого показателя '
                  f'(корреляция {tf.ru_num(r["corr"], 2)}). В {tf.ru_num(100 * r["share_gt_0.2"], 0)}% месяцев ошибка больше 0,2 п.п. — '
                  f'а это уже {tf.ru_num(12 * 0.2, 1)} п.п. в годовом выражении. Вывод: недельные данные надёжно показывают '
                  f'направление и порядок месячной инфляции, но точную цифру месяца заменить не могут.\n\n'
                  f'Почему ошибка неизбежна: недельный индекс — цены на дату регистрации (обычно понедельник), а неделя может '
                  f'захватывать соседний месяц; набор товаров в недельном мониторинге уже, чем в месячном ИПЦ; месячные данные '
                  f'учитывают тарифы и услуги, которые в недельных представлены не полностью.',
            'en': f'**How significant the error is (since {START[:4]}).** The typical error (RMSE) is **{tf.ru_num(r["rmse"], 2)} p.p.**, '
                  f'the median {tf.ru_num(r["median"], 2)} p.p.; {verdict}. For comparison: m/m inflation varies with a standard '
                  f'deviation of {tf.ru_num(r["std_ref"], 2)} p.p. around an average of {tf.ru_num(ref_s.mean(), 2)}%, '
                  f'so the error is **{tf.ru_num(100 * r["rmse_to_std"], 0)}%** of the spread of the indicator itself '
                  f'(correlation {tf.ru_num(r["corr"], 2)}). In {tf.ru_num(100 * r["share_gt_0.2"], 0)}% of months the error exceeds 0.2 p.p. — '
                  f'that is already {tf.ru_num(12 * 0.2, 1)} p.p. annualised. Conclusion: weekly data reliably show the direction '
                  f'and order of magnitude of monthly inflation, but cannot replace the exact monthly number.\n\n'
                  f'Why the error is unavoidable: the weekly index is prices on the registration date (usually Monday), and a week '
                  f'can straddle two months; the weekly basket is narrower than the monthly CPI; monthly data include tariffs and '
                  f'services that are only partly covered weekly.'})
    ww, yt = tr('н/н', 'w/w'), tr('с начала года', 'year to date')
    tf.plot_series({ww: x, yt: tf.ytd(I, 'W')}, N(t), '%', bars=[ww], right=[yt], start='2024', nd=2)

In [ ]:
show_md({
"ru": r"""
### 6.9 Прирост к концу прошлого года (ytd)

Денежная масса M2X (без валютной переоценки) «с начала года». Информация та же, что в м/м:
$1+m_t = (1+ytd_t)/(1+ytd_{t-1})$ внутри года, в январе $m = ytd$. Значит, из ytd получается всё, что из м/м.

Два графика «по годам» (по x — месяцы, по линии на год); к каждому — вариант «последние два года на фоне диапазона»:
* **уровень** — цепной индекс: видно, как каждый месяц соотносится с тем же месяцем других лет;
* **путь внутри года** — каждый год нормирован так, что конец предыдущего года = 100, январь = 100·(1 + рост за январь),
  февраль = 100·(1 + рост за январь–февраль) и т.д. Так годы сравниваются в относительном выражении независимо от уровня.

Выпадает ли первый год? Для ряда **уровней** — да: нормировать первый год не на что, уровня декабря предыдущего года нет
(`tf.year_profile` такой год отбрасывает). Но здесь исходный ряд — ytd, и он уже отсчитан от декабря: цепной индекс,
собранный из него, неявно начинается с «декабрь перед первым годом = 100». Добавляем эту точку явно — и первый год
остаётся. По сути нормированный профиль здесь и есть опубликованный ряд `100 + ytd`; проверка ниже это подтверждает.
""",
"en": r"""
### 6.9 Growth vs end of last year (ytd)

Money supply M2X (excluding FX revaluation) "since the start of the year". It carries the same information as m/m:
$1+m_t = (1+ytd_t)/(1+ytd_{t-1})$ within the year, and in January $m = ytd$. So ytd gives everything m/m gives.

Two "by year" charts (months on the x axis, one line per year), each with a "last two years against the range" variant:
* **level** — a chain index: shows how each month compares with the same month of other years;
* **path within the year** — each year is normalised so that the end of the previous year = 100, January = 100·(1 + growth
  in January), February = 100·(1 + growth in January–February) and so on. Years are compared in relative terms regardless of level.

Does the first year drop out? For a series of **levels** — yes: there is nothing to normalise the first year by, the level of
the previous December is missing (`tf.year_profile` drops such a year). But here the source series is ytd, already measured
from December: the chain index built from it implicitly starts at "December before the first year = 100". We add that point
explicitly — and the first year stays. In fact the normalised profile here is the published `100 + ytd`; the check below confirms it.
"""})

In [ ]:
tf.set_section('6.9', {'ru': 'Прирост к концу прошлого года', 'en': 'Growth vs end of last year'})
t = need('ytd_growth')
if t:
    c = C(t)
    f = c['freq']
    tf.explain(c, SER[t])
    x = tf.growth_x(SER[t])
    m_ = tf.mom_from_ytd(x)
    I = tf.chain(m_)
    print(tr('Обратная проверка ytd → м/м → ytd:', 'Round trip ytd → m/m → ytd:'), tf.same(tf.ytd(I, f), x))
    mr = tr('м/м (восстановлено)', 'm/m (recovered)')
    tf.plot_series({tr('с начала года (как опубликовано)', 'year to date (as published)'): x, mr: m_,
                    tr('г/г (восстановлено)', 'y/y (recovered)'): tf.yoy(I, f)},
                   N(t), '%', bars=[mr], start='2021', nd=1)

    # Chain index with an explicit base: December before the first observation = 100
    # Цепной индекс с явной базой: декабрь перед первым наблюдением = 100
    I0 = pd.concat([pd.Series([100.0], index=[I.dropna().index[0] - 1]), I.dropna()])
    lvl_prof = tf.year_profile(I0, f)
    tf.plot_year_profile(lvl_prof, f, tr(f'{N(t)}: уровень по месяцам', f'{N(t)}: level by month'),
                         tr('цепной индекс (декабрь перед первым годом = 100); каждая линия — год',
                            'chain index (December before the first year = 100); one line per year'), nd=0)
    tf.plot_year_range(lvl_prof, f, tr(f'{N(t)}: уровень, последние два года на фоне прошлых лет',
                                       f'{N(t)}: level, the last two years against past years'),
                       tr('цепной индекс; область — мин–макс по каждому месяцу за более ранние годы',
                          'chain index; band — min–max for each month over earlier years'), nd=0)
    prof = tf.year_profile(I0, f, normalize=True)
    print(tr('Нормированный профиль совпадает с опубликованным ytd (100 + ytd):',
             'The normalised profile matches the published ytd (100 + ytd):'),
          tf.same(prof.drop(index=0).unstack().reset_index(drop=True), (100 + x).dropna().reset_index(drop=True)))
    tf.plot_year_profile(prof, f, tr(f'{N(t)}: путь внутри года', f'{N(t)}: path within the year'),
                         tr('конец предыдущего года = 100; январь = 100·(1 + рост за январь), февраль — за январь–февраль и т.д.',
                            'end of previous year = 100; January = 100·(1 + growth in January), February — January–February, etc.'),
                         hline=100)
    tf.plot_year_range(prof, f, tr(f'{N(t)}: путь внутри года, последние два года на фоне прошлых лет',
                                   f'{N(t)}: path within the year, the last two years against past years'),
                       tr('конец предыдущего года = 100; область — мин–макс по каждому месяцу за более ранние годы',
                          'end of previous year = 100; band — min–max for each month over earlier years'), hline=100)

In [ ]:
show_md({
"ru": r"""
### 6.10 Прирост к тому же периоду прошлого года

Промпроизводство г/г — тупиковая размерность (раздел 4.2): ни м/м, ни уровня, ни квартального роста — только сгладить.
Сглаживание в конце ряда — **только нецентрированное**: центрированное окно в последних точках опирается на неполные
данные и пересматривается задним числом.
""",
"en": r"""
### 6.10 Growth vs same period last year

Industrial production y/y is a dead-end dimension (section 4.2): no m/m, no level, no quarterly growth — it can only be
smoothed. At the end of the series smoothing must be **trailing only**: a centred window relies on incomplete data at the
last points and gets revised retroactively.
"""})

In [ ]:
tf.set_section('6.10', {'ru': 'Прирост г/г', 'en': 'Growth y/y'})
t = need('yoy_growth')
if t:
    c = C(t)
    f = c['freq']
    tf.explain(c, SER[t])
    x = tf.growth_x(SER[t])
    trail = x.rolling(3, min_periods=3).mean()
    center = x.rolling(5, center=True, min_periods=5).mean()
    print(tr(f'Центрированное 5 мес. заканчивается в {center.dropna().index[-1]} — на 2 месяца раньше ряда; '
             f'нецентрированное 3 мес.: {tf.pct(trail.iloc[-1])}',
             f'The centred 5-month average ends in {center.dropna().index[-1]} — 2 months before the series; '
             f'trailing 3-month: {tf.pct(trail.iloc[-1])}'))
    cc_ = tr('центрированное 5 мес.', 'centred 5 months')
    tf.plot_series({tr('г/г', 'y/y'): x, tr('среднее 3 мес.', '3-month average'): trail, cc_: center}, N(t), '%',
                   start='2021', hline=0, dashed=[cc_])

In [ ]:
show_md({
"ru": r"""
### 6.11 С начала года к тому же периоду прошлого года

Инвестиции «январь–июнь к январю–июню прошлого года» (здесь — квартальные). Каждое значение включает все предыдущие
кварталы года, поэтому к концу года ряд всё меньше реагирует на текущий квартал. Полезные точки — 4 квартал (рост за год)
и 2 квартал (полугодие). Квартальный прирост без уровней базового года не извлечь.
""",
"en": r"""
### 6.11 Year-to-date vs same period last year

Investment "January–June vs January–June last year" (quarterly here). Each value includes all earlier quarters of the year,
so towards year-end the series reacts less and less to the current quarter. The useful points are Q4 (growth over the year)
and Q2 (the half-year). Quarterly growth cannot be extracted without the levels of a base year.
"""})

In [ ]:
tf.set_section('6.11', {'ru': 'С начала года к прошлому году', 'en': 'Year-to-date vs last year'})
t = need('cmlt_cmlt')
if t:
    c = C(t)
    f = c['freq']
    tf.explain(c, SER[t])
    x = tf.growth_x(SER[t])
    yr = x[(x.index.month == 12) if f == 'M' else (x.index.quarter == 4)]
    display(pd.Series(yr.values, index=yr.index.year).tail(8).round(1).to_frame(tr('рост за год, %', 'growth over the year, %')).T)
    tf.season_chart(x, f, N(t), tr('%; каждая линия — год, последняя точка года — рост за год',
                                   '%; one line per year, the last point of a year is growth over the year'), nd=1)
    tf.plot_year_range(x, f, tr(f'{N(t)}: последние два года на фоне прошлых лет', f'{N(t)}: the last two years against past years'),
                       tr('%; область — мин–макс по каждому кварталу за более ранние годы',
                          '%; band — min–max for each quarter over earlier years'), hline=0, nd=1)

In [ ]:
show_md({
"ru": r"""
### 6.12 Доля, %

Безработица — уже отношение. Изменения — в п.п., в квартал — среднее.
""",
"en": r"""
### 6.12 Share, %

Unemployment is already a ratio. Changes are in p.p.; to a quarter — the average.
"""})

In [ ]:
tf.set_section('6.12', {'ru': 'Доля', 'en': 'Share'})
t = need('share')
if t:
    c = C(t)
    f = c['freq']
    tf.explain(c, SER[t])
    x = SER[t]
    ch = tr('изменение за год, п.п.', 'change over the year, p.p.')
    tf.plot_series({tr('уровень, %', 'level, %'): x, ch: tf.diff_pp(x, tf.PPY[f])}, N(t), tr('%; п.п.', '%; p.p.'),
                   bars=[ch], start=START, hline=0)

In [ ]:
show_md({
"ru": r"""
### 6.13 Ставка, % годовых

Ключевая ставка — дневной ряд. В месяц — средним (средняя стоимость денег) или на конец.
Изменение ставки — в п.п.; «ставка г/г в процентах» посчитать можно, но смысла нет — это пример категории ◇.

**Реальная ставка** = номинальная ставка − **инфляционные ожидания** (ex ante): решение о сбережении или кредите
принимается с оглядкой на будущую, а не прошлую инфляцию. Ставка минус фактическая инфляция г/г — лишь грубая оценка
ex post, поэтому здесь её не строим: в наборе нет ряда ожиданий (опросы ЦБ / инФОМ или рыночные оценки по ОФЗ-ИН).
""",
"en": r"""
### 6.13 Rate, % per annum

The key rate is a daily series. To a month — the average (the average cost of money) or the end value.
A change of the rate is in p.p.; "the rate y/y in percent" is computable but meaningless — an example of the ◇ category.

**The real rate** = nominal rate − **inflation expectations** (ex ante): saving and borrowing decisions look at future, not
past, inflation. The rate minus actual y/y inflation is only a rough ex post proxy, so it is not built here: the data set
has no expectations series (Bank of Russia / inFOM surveys or market estimates from inflation-linked OFZ).
"""})

In [ ]:
tf.set_section('6.13', {'ru': 'Ставка', 'en': 'Rate'})
t = need('rate')
if t:
    c = C(t)
    tf.explain(c, SER[t])
    rm = tf.aggregate(SER[t], 'M', 'mean', 'D', full_only=False)
    re_ = tf.aggregate(SER[t], 'M', 'last', 'D', full_only=False)
    em = tr('на конец месяца', 'end of month')
    tf.plot_series({tr('в среднем за месяц', 'monthly average'): rm, em: re_}, N(t), tr('% годовых', '% per annum'),
                   start='2019', dashed=[em])
    ok_, bad_ = tr('изменение за год, п.п. (верно)', 'change over the year, p.p. (right)'), \
        tr('«г/г», % (посчитать можно, смысла нет)', '"y/y", % (computable, meaningless)')
    tf.plot_series({ok_: tf.diff_pp(rm, 12), bad_: tf.yoy(rm, 'M')},
                   tr('Изменение ставки — в п.п., а не в процентах', 'A rate changes in p.p., not in percent'),
                   tr('среднемесячная ставка', 'monthly average rate'), bars=[ok_], right=[bad_], start='2019', hline=0)

In [ ]:
show_md({
"ru": r"""
## 7. Свой расчёт: любой тикер, любая функция

Типы показателей нужны только для подбора примеров. Функции модуля принимают **явные аргументы** — частоту, тип ряда,
валюту — и считают, не спрашивая, «можно ли». Достаточно знать характеристики ряда. Ответственность за смысл — на вас;
`tf.available` подскажет, если сомневаетесь.

Шаблон: экспорт в рублях по среднему курсу, рост г/г в рублях и в долларах — разница и есть вклад курса.
""",
"en": r"""
## 7. Your own calculation: any ticker, any function

Indicator types are only needed to pick examples. The module's functions take **explicit arguments** — frequency, series
type, currency — and compute without asking "is it allowed". Knowing the characteristics of the series is enough. The
responsibility for the meaning is yours; `tf.available` will hint if in doubt.

Template: exports in roubles at the average rate, y/y growth in roubles and in dollars — the difference is the exchange-rate contribution.
"""})

In [ ]:
tf.set_section('7', {'ru': 'Свой расчёт', 'en': 'Your own calculation'})
TICKER = 'RU_EXPG_Q_CB'                    # any ticker from CHARS | любой тикер из CHARS
if load(TICKER) is not None:
    c = C(TICKER)
    s = SER[TICKER]
    usd = s * c['mult']                                    # in dollars | в долларах
    rub = tf.convert(usd, 'USD', 'RUB', 'Q', 'mean')       # explicit: USD -> RUB, quarterly, average rate | явно
    usd4, rub4 = tf.rolling(usd, 'Q', 'Y', 'sum'), tf.rolling(rub, 'Q', 'Y', 'sum')
    tf.plot_series({tr('в долларах', 'in dollars'): tf.yoy(usd4, 'Q'), tr('в рублях', 'in roubles'): tf.yoy(rub4, 'Q')},
                   tr(f'{N(TICKER)}: рост в рублях и в долларах', f'{N(TICKER)}: growth in roubles and in dollars'),
                   tr('4 квартала к 4 кварталам, %; разница — курс', '4 quarters vs 4 quarters, %; the gap is the exchange rate'),
                   start=START, hline=0)
    display(tf.avail_table(c))

In [ ]:
show_md({
"ru": r"""
## 8. Шпаргалка

**Что из чего получается**

| лежит в данных | восстанавливается | можно получить | посчитать можно, смысла нет |
|---|---|---|---|
| уровень (`none`) | — | всё: приросты, суммы, валюта, %ВВП | запас — накапливать; поток — к декабрю |
| накопленный (`cmlt`) | уровень **точно** | всё, что из уровня | — |
| м/м, к/к, н/н, к декабрю | уровень **с точностью до множителя** | любые приросты, cmlt/cmlt | валюта, %ВВП для неденежных рядов |
| г/г, cmlt/cmlt | **ничего** без базового года | сглаживание, годовой рост (декабрь / 4 кв. cmlt/cmlt) | сглаживание cmlt/cmlt |
| %, ставка, %ВВП | — | изменения в п.п., средние | приросты в %, валюта, накопление |
| номинальный ↔ реальный | через индекс цен | реальный = номинальный / дефлятор | номинальный ряд к реальному ВВП |

**Чек-лист перед расчётом**
1. Посмотреть характеристики: `metric`, `transf`, `stock_flow`, `dim`, `currency`, `mult`, `freq`, `real_nominal`.
2. Прирост хранится как `0,4` или как `100,4`?
3. Перед делением рядов — оба в абсолютные единицы (`× mult`), одну валюту и одинаково номинальные/реальные.
4. Курс: поток — средний, запас — на конец периода, накопленный — помесячно.
5. Агрегация: поток — сумма полных периодов, запас — значение на конец, цены — среднее.
6. К ВВП: поток — квартал/квартал или 4 кв./4 кв.; запас — только к ВВП за 4 кв.

**Типичные ошибки**
* накопленный с начала года поток делят на курс последнего месяца;
* запас относят к ВВП одного квартала (завышение в 4 раза и ложная сезонность);
* суммируют неполный последний квартал;
* путают млн и млрд (`mult`) при отношении к ВВП;
* сравнивают номинальный рост с реальным (номинальная выручка +10% при инфляции 9% — это +1% реально);
* аннуализируют несезонно сглаженный м/м;
* сглаживают г/г центрированным окном — последние точки пересматриваются;
* изменение доли или ставки выражают в процентах, а не в п.п.;
* реальную ставку считают как ставку минус прошлую инфляцию вместо инфляционных ожиданий.
""",
"en": r"""
## 8. Cheat sheet

**What can be derived from what**

| stored in the data | recoverable | can be derived | computable but meaningless |
|---|---|---|---|
| level (`none`) | — | everything: growth, sums, currency, % of GDP | cumulating a stock; a flow vs December |
| cumulative (`cmlt`) | the level **exactly** | everything the level gives | — |
| m/m, q/q, w/w, vs December | the level **up to a factor** | any growth rates, cmlt/cmlt | currency, % of GDP for non-money series |
| y/y, cmlt/cmlt | **nothing** without a base year | smoothing, annual growth (December / Q4 of cmlt/cmlt) | smoothing cmlt/cmlt |
| %, rate, % of GDP | — | changes in p.p., averages | growth in %, currency, cumulation |
| nominal ↔ real | via a price index | real = nominal / deflator | a nominal series over real GDP |

**Checklist before a calculation**
1. Look at the characteristics: `metric`, `transf`, `stock_flow`, `dim`, `currency`, `mult`, `freq`, `real_nominal`.
2. Is growth stored as `0.4` or as `100.4`?
3. Before dividing series — both in absolute units (`× mult`), one currency, both nominal or both real.
4. FX rate: a flow — average, a stock — end of period, a cumulative series — month by month.
5. Aggregation: a flow — sum of complete periods, a stock — the end value, prices — the average.
6. To GDP: a flow — quarter/quarter or 4Q/4Q; a stock — only to 4-quarter GDP.

**Typical mistakes**
* a year-to-date flow divided by the last month's FX rate;
* a stock divided by one quarter's GDP (4 times too high and falsely seasonal);
* summing an incomplete last quarter;
* mixing up millions and billions (`mult`) in a ratio to GDP;
* comparing nominal growth with real growth (nominal revenue +10% with 9% inflation is +1% in real terms);
* annualising a non-seasonally-adjusted m/m;
* smoothing y/y with a centred window — the last points get revised;
* expressing a change of a share or rate in percent instead of p.p.;
* computing the real rate as the rate minus past inflation instead of inflation expectations.
"""})